# PA3 — Parte 2: curva de orçamento

1. Ative **GPU** e **Internet** no Kaggle.
2. Adicione o Dataset `pa3_dados` completo, como na Parte 1.
3. Adicione também `pa3_resultados_completos.zip` da Parte 1 como Input. Pode estar extraído:
   o notebook também aceita a pasta `resultados/trilha_a_completa`.
4. Execute **Run All**. Os scripts já estão incorporados.

São **10 treinos supervisionados**: piso ImageNet e método ImageNet+SimCLR,
com 1, 5 e 20 rótulos por classe. Seeds 42,43,44 para 5; seed 42 nos demais.
O checkpoint SimCLR é fixo; a variabilidade medida é a do fine-tuning.
Teto do enunciado: **0,686**, sem treino adicional.
Os CSVs oficiais são mantidos; val escolhe a época, test_public só fornece as métricas finais.

Salve os outputs no Kaggle. Se precisar retomar em outra sessão, anexe os outputs salvos e
preencha `RESTORE_FROM` abaixo. A retomada é por época e também pula treinos já concluídos.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess
import torch
assert torch.cuda.is_available(), 'Ative a GPU antes de continuar.'
print(torch.__version__, torch.cuda.get_device_name(0))
WORK = Path('/kaggle/working/pa3_parte2')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
DATA_OVERRIDE = None
PART1_ZIP_OVERRIDE = None
RESTORE_FROM = None  # Pasta parte2 de outputs salvos de uma execução anterior.
INPUT = Path('/kaggle/input')
roots = sorted({p.parent for p in INPUT.rglob('classes.csv')
    if (p.parent/'labels/rotulos_20.csv').is_file() and (p.parent/'images/train').is_dir()})
if DATA_OVERRIDE:
    DATA = Path(DATA_OVERRIDE)
else:
    assert len(roots)==1, f'Preencha DATA_OVERRIDE. Pastas encontradas: {roots}'
    DATA = roots[0]
OUTPUT = WORK/'resultados/parte2'
if RESTORE_FROM:
    restore = Path(RESTORE_FROM)
    assert restore.is_dir() and restore.resolve()!=OUTPUT.resolve()
    shutil.copytree(restore, OUTPUT, dirs_exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
zips = list(INPUT.rglob('pa3_resultados_completos.zip'))
PART1_ZIP = Path(PART1_ZIP_OVERRIDE) if PART1_ZIP_OVERRIDE else (zips[0] if len(zips)==1 else None)
EXTRA_ARGS = []
if PART1_ZIP:
    EXTRA_ARGS = ['--part1-zip', str(PART1_ZIP)]
else:
    reports = list(INPUT.rglob('relatorio_trilha_a.json'))
    assert len(reports)==1, 'Adicione o ZIP da Parte 1 ou sua pasta extraída; se houver vários ZIPs, preencha PART1_ZIP_OVERRIDE.'
    report = json.loads(reports[0].read_text())
    assert not report.get('preliminary', True)
    checkpoint = reports[0].parent/report['selected_augmentation']/'imagenet_ssl.pt'
    assert checkpoint.is_file(), checkpoint
    EXTRA_ARGS = ['--ssl-checkpoint', str(checkpoint)]
print('Dados:', DATA, '\nParte 1:', EXTRA_ARGS)


In [ ]:
SCRIPTS = {'part0.py': '"""Supervised baselines for Part 0 of the PA3 assignment."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport random\nfrom pathlib import Path\nfrom typing import Any\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageOps\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\n\n\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\ndef load_classes(classes_csv: Path) -> list[str]:\n    with classes_csv.open(encoding="utf-8-sig", newline="") as file:\n        rows = list(csv.DictReader(file))\n    if not rows or not {"idx", "label"}.issubset(rows[0]):\n        raise ValueError(f"{classes_csv} deve conter as colunas idx,label.")\n    rows.sort(key=lambda row: int(row["idx"]))\n    indices = [int(row["idx"]) for row in rows]\n    if indices != list(range(len(rows))):\n        raise ValueError(f"Os índices em {classes_csv} devem começar em zero e ser contíguos.")\n    return [row["label"] for row in rows]\n\n\ndef read_labels(labels_csv: Path, class_names: list[str]) -> list[tuple[str, int]]:\n    name_to_index = {name: index for index, name in enumerate(class_names)}\n    with labels_csv.open(encoding="utf-8-sig", newline="") as file:\n        reader = csv.DictReader(file)\n        if not {"image_id", "label"}.issubset(reader.fieldnames or []):\n            raise ValueError(f"{labels_csv} deve conter as colunas image_id,label.")\n        rows: list[tuple[str, int]] = []\n        seen_ids: set[str] = set()\n        for line, row in enumerate(reader, start=2):\n            image_id = row["image_id"].strip()\n            raw_label = row["label"].strip()\n            if not image_id or not raw_label:\n                raise ValueError(f"ID ou rótulo vazio em {labels_csv}, linha {line}.")\n            if image_id in seen_ids:\n                raise ValueError(f"ID duplicado {image_id!r} em {labels_csv}.")\n            seen_ids.add(image_id)\n            if image_id.lower().endswith(".jpg"):\n                image_id = image_id[:-4]\n            try:\n                label_index = int(raw_label)\n            except ValueError:\n                if raw_label not in name_to_index:\n                    raise ValueError(f"Classe desconhecida {raw_label!r} em {labels_csv}, linha {line}.")\n                label_index = name_to_index[raw_label]\n            if not 0 <= label_index < len(class_names):\n                raise ValueError(f"Índice de classe inválido {label_index} em {labels_csv}, linha {line}.")\n            rows.append((image_id, label_index))\n    if not rows:\n        raise ValueError(f"Nenhum exemplo encontrado em {labels_csv}.")\n    return rows\n\n\nclass CameraTrapDataset(Dataset[tuple[torch.Tensor, int]]):\n    def __init__(self, images_dir: Path, labels_csv: Path, class_names: list[str], image_size: int):\n        self.images_dir = images_dir\n        self.rows = read_labels(labels_csv, class_names)\n        self.image_size = image_size\n        self.mean = torch.tensor(MEAN, dtype=torch.float32).view(3, 1, 1)\n        self.std = torch.tensor(STD, dtype=torch.float32).view(3, 1, 1)\n        missing = [\n            image_id\n            for image_id, _ in self.rows\n            if not (images_dir / f"{image_id}.jpg").is_file()\n        ]\n        if missing:\n            sample = ", ".join(missing[:5])\n            raise FileNotFoundError(\n                f"{len(missing)} imagens referenciadas por {labels_csv} não estão em "\n                f"{images_dir}; exemplos: {sample}"\n            )\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:\n        image_id, label = self.rows[index]\n        with Image.open(self.images_dir / f"{image_id}.jpg") as source:\n            image = ImageOps.pad(\n                source.convert("RGB"),\n                (self.image_size, self.image_size),\n                method=Image.Resampling.BILINEAR,\n            )\n        array = np.asarray(image, dtype=np.float32) / 255.0\n        tensor = torch.from_numpy(array).permute(2, 0, 1)\n        return (tensor - self.mean) / self.std, label\n\n\nclass SmallCNN(nn.Module):\n    """A compact CNN trained from random initialization."""\n\n    def __init__(self, num_classes: int):\n        super().__init__()\n        self.features = nn.Sequential(\n            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(32),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(64),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(128),\n            nn.ReLU(inplace=True),\n            nn.AdaptiveAvgPool2d(1),\n        )\n        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, num_classes))\n\n    def forward(self, images: torch.Tensor) -> torch.Tensor:\n        return self.classifier(self.features(images))\n\n\ndef make_model(model_name: str, num_classes: int) -> nn.Module:\n    if model_name == "small_cnn":\n        return SmallCNN(num_classes)\n    if model_name == "resnet18":\n        try:\n            from torchvision.models import ResNet18_Weights, resnet18\n        except ImportError as error:\n            raise RuntimeError(\n                "A versão ResNet-18 requer torchvision. Instale as dependências com "\n                "`python -m pip install -r requirements.txt`."\n            ) from error\n        model = resnet18(weights=ResNet18_Weights.DEFAULT)\n        model.fc = nn.Linear(model.fc.in_features, num_classes)\n        return model\n    raise ValueError(f"Modelo desconhecido: {model_name}")\n\n\ndef evaluate(\n    model: nn.Module, loader: DataLoader[tuple[torch.Tensor, int]], device: torch.device\n) -> tuple[float, float, float, list[int], list[int]]:\n    criterion = nn.CrossEntropyLoss()\n    model.eval()\n    total_loss = 0.0\n    actual: list[int] = []\n    predicted: list[int] = []\n    with torch.inference_mode():\n        for images, labels in loader:\n            images = images.to(device)\n            labels = labels.to(device)\n            logits = model(images)\n            total_loss += criterion(logits, labels).item() * labels.size(0)\n            actual.extend(labels.cpu().tolist())\n            predicted.extend(logits.argmax(dim=1).cpu().tolist())\n    return (\n        total_loss / len(loader.dataset),\n        f1_score(actual, predicted, average="macro", zero_division=0),\n        accuracy_score(actual, predicted),\n        actual,\n        predicted,\n    )\n\n\ndef write_history_plot(history: dict[str, list[float]], path: Path) -> None:\n    epochs = range(1, len(history["train_loss"]) + 1)\n    figure, axes = plt.subplots(1, 2, figsize=(11, 4))\n    axes[0].plot(epochs, history["train_loss"], label="Treino")\n    axes[0].plot(epochs, history["val_loss"], label="Validação")\n    axes[0].set(title="Perda", xlabel="Época", ylabel="Cross-entropy")\n    axes[1].plot(epochs, history["train_macro_f1"], label="Treino")\n    axes[1].plot(epochs, history["val_macro_f1"], label="Validação")\n    axes[1].set(title="F1 macro", xlabel="Época", ylabel="F1 macro")\n    for axis in axes:\n        axis.legend()\n        axis.grid(alpha=0.25)\n    figure.tight_layout()\n    figure.savefig(path, dpi=160)\n    plt.close(figure)\n\n\ndef train_model(\n    model_name: str,\n    datasets: dict[str, CameraTrapDataset],\n    class_names: list[str],\n    output_dir: Path,\n    epochs: int,\n    batch_size: int,\n    image_size: int,\n    seed: int,\n) -> dict[str, Any]:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = make_model(model_name, len(class_names)).to(device)\n    if model_name == "resnet18":\n        learning_rate = 1e-4\n    else:\n        learning_rate = 1e-3\n    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)\n    criterion = nn.CrossEntropyLoss()\n    loaders = {\n        split: DataLoader(\n            dataset,\n            batch_size=batch_size,\n            shuffle=(split == "train"),\n            num_workers=0,\n            pin_memory=device.type == "cuda",\n        )\n        for split, dataset in datasets.items()\n    }\n    output_dir.mkdir(parents=True, exist_ok=True)\n    history: dict[str, list[float]] = {\n        "train_loss": [],\n        "val_loss": [],\n        "train_macro_f1": [],\n        "val_macro_f1": [],\n    }\n    best_f1 = -1.0\n    best_val_loss = float("inf")\n    best_checkpoint = output_dir / "best.pt"\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        for images, labels in loaders["train"]:\n            images = images.to(device)\n            labels = labels.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = criterion(model(images), labels)\n            loss.backward()\n            optimizer.step()\n\n        train_loss, train_f1, _, _, _ = evaluate(model, loaders["train"], device)\n        val_loss, val_f1, _, _, _ = evaluate(model, loaders["val"], device)\n        history["train_loss"].append(train_loss)\n        history["val_loss"].append(val_loss)\n        history["train_macro_f1"].append(train_f1)\n        history["val_macro_f1"].append(val_f1)\n        print(\n            f"{model_name} época {epoch:03d}/{epochs} | "\n            f"perda treino/val={train_loss:.4f}/{val_loss:.4f} | "\n            f"F1 macro treino/val={train_f1:.4f}/{val_f1:.4f}"\n        )\n        if val_f1 > best_f1 or (val_f1 == best_f1 and val_loss < best_val_loss):\n            best_f1 = val_f1\n            best_val_loss = val_loss\n            torch.save(\n                {\n                    "model_name": model_name,\n                    "model_state_dict": model.state_dict(),\n                    "class_names": class_names,\n                    "image_size": image_size,\n                    "mean": MEAN,\n                    "std": STD,\n                    "seed": seed,\n                    "best_epoch": epoch,\n                },\n                best_checkpoint,\n            )\n\n    with (output_dir / "history.json").open("w", encoding="utf-8") as file:\n        json.dump(history, file, indent=2)\n    if model_name == "resnet18":\n        write_history_plot(history, output_dir / "curva_treino_validacao.png")\n\n    checkpoint = torch.load(best_checkpoint, map_location=device, weights_only=True)\n    model.load_state_dict(checkpoint["model_state_dict"])\n    test_loss, test_f1, test_accuracy, actual, predicted = evaluate(\n        model, loaders["test_public"], device\n    )\n    metrics = {\n        "model": model_name,\n        "best_epoch": checkpoint["best_epoch"],\n        "test_public_macro_f1": test_f1,\n        "test_public_accuracy": test_accuracy,\n        "test_public_loss": test_loss,\n        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),\n        "num_test_public": len(actual),\n        "class_names": class_names,\n    }\n    with (output_dir / "test_public_metrics.json").open("w", encoding="utf-8") as file:\n        json.dump(metrics, file, indent=2)\n    print(\n        f"{model_name} | melhor época pelo val: {checkpoint[\'best_epoch\']} | "\n        f"F1 macro test_public={test_f1:.4f} | acurácia test_public={test_accuracy:.4f}"\n    )\n    if model_name == "resnet18":\n        matrix = confusion_matrix(actual, predicted, labels=list(range(len(class_names))))\n        with (output_dir / "matriz_confusao.csv").open(\n            "w", encoding="utf-8", newline=""\n        ) as file:\n            writer = csv.writer(file)\n            writer.writerow(["classe", *class_names])\n            writer.writerows(\n                [class_names[index], *row.tolist()] for index, row in enumerate(matrix)\n            )\n        figure, axis = plt.subplots(figsize=(12, 10))\n        image = axis.imshow(matrix, cmap="Blues")\n        axis.set(\n            xticks=range(len(class_names)),\n            yticks=range(len(class_names)),\n            xticklabels=class_names,\n            yticklabels=class_names,\n            xlabel="Classe prevista",\n            ylabel="Classe real",\n            title="ResNet-18 — matriz de confusão no test_public",\n        )\n        plt.setp(axis.get_xticklabels(), rotation=60, ha="right", rotation_mode="anchor")\n        figure.colorbar(image, ax=axis)\n        figure.tight_layout()\n        figure.savefig(output_dir / "matriz_confusao.png", dpi=160)\n        plt.close(figure)\n    return metrics\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Treina as duas baselines supervisionadas da Parte 0."\n    )\n    parser.add_argument("--data-dir", type=Path, default=Path("pa3_dados"))\n    parser.add_argument("--output-dir", type=Path, default=Path("parte0/resultados"))\n    parser.add_argument("--epochs", type=int, default=30)\n    parser.add_argument("--batch-size", type=int, default=16)\n    parser.add_argument("--image-size", type=int, default=224)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument(\n        "--model",\n        choices=("both", "small_cnn", "resnet18"),\n        default="both",\n        help="Permite executar uma versão individual ou as duas em sequência.",\n    )\n    args = parser.parse_args()\n    if args.epochs < 1 or args.batch_size < 1 or args.image_size < 32:\n        parser.error("--epochs e --batch-size devem ser positivos; --image-size deve ser >= 32.")\n    return args\n\n\ndef main() -> None:\n    args = parse_args()\n    data_dir: Path = args.data_dir\n    classes = load_classes(data_dir / "classes.csv")\n    split_files = {\n        "train": (data_dir / "images" / "train", data_dir / "labels" / "rotulos_5.csv"),\n        "val": (data_dir / "images" / "val", data_dir / "labels" / "val.csv"),\n        "test_public": (\n            data_dir / "images" / "test_public",\n            data_dir / "labels" / "test_public.csv",\n        ),\n    }\n    missing = [str(labels) for _, labels in split_files.values() if not labels.is_file()]\n    missing.extend(str(images) for images, _ in split_files.values() if not images.is_dir())\n    if missing:\n        raise FileNotFoundError(\n            "Dados necessários ausentes: "\n            + ", ".join(missing)\n            + ". Extraia o pacote completo pa3_dados, incluindo labels/ e images/val."\n        )\n    datasets = {\n        split: CameraTrapDataset(images, labels, classes, args.image_size)\n        for split, (images, labels) in split_files.items()\n    }\n    if len(datasets["train"]) != 105:\n        raise ValueError(\n            f"A Parte 0 requer os 105 exemplos de rotulos_5.csv; "\n            f"foram encontrados {len(datasets[\'train\'])}."\n        )\n    if len(datasets["val"]) != 202:\n        print(f"Aviso: o val tem {len(datasets[\'val\'])} exemplos (o pacote informa 202).")\n    model_names = ("small_cnn", "resnet18") if args.model == "both" else (args.model,)\n    results = []\n    for model_name in model_names:\n        results.append(\n            train_model(\n                model_name,\n                datasets,\n                classes,\n                args.output_dir / model_name,\n                args.epochs,\n                args.batch_size,\n                args.image_size,\n                args.seed,\n            )\n        )\n    with (args.output_dir / "resumo_test_public.json").open("w", encoding="utf-8") as file:\n        json.dump(results, file, indent=2)\n\n\nif __name__ == "__main__":\n    main()\n', 'parte2.py': '"""Budget curves: matched ImageNet and ImageNet+SimCLR fine-tuning runs."""\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import Counter\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport sys\nimport zipfile\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\nfrom sklearn.metrics import accuracy_score, f1_score\nfrom torch import nn\nfrom torch.utils.data import DataLoader\nfrom torchvision.models import resnet18, ResNet18_Weights\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent / \'parte0\'))\nfrom part0 import CameraTrapDataset, load_classes, MEAN, STD\n\nCEILING = 0.686\nSEEDS = (42, 43, 44)\n\n\ndef write_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef write_csv(path, rows):\n    with path.open(\'w\', newline=\'\', encoding=\'utf-8\') as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with path.open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef locate_ssl(args):\n    if args.ssl_checkpoint:\n        return args.ssl_checkpoint\n    assert args.part1_zip and args.part1_zip.is_file(), \'Informe --part1-zip ou --ssl-checkpoint.\'\n    with zipfile.ZipFile(args.part1_zip) as archive:\n        report = json.loads(archive.read(\'resultados/trilha_a_completa/relatorio_trilha_a.json\'))\n        assert not report.get(\'preliminary\', True), \'A Parte 1 precisa estar completa.\'\n        crop = report[\'selected_augmentation\']\n        target = args.output_dir / \'source\' / crop / \'imagenet_ssl.pt\'\n        target.parent.mkdir(parents=True, exist_ok=True)\n        # Read one known member; never extract arbitrary ZIP paths.\n        target.write_bytes(archive.read(f\'resultados/trilha_a_completa/{crop}/imagenet_ssl.pt\'))\n        write_json(args.output_dir / \'source\' / \'parte1_report.json\', report)\n    return target\n\n\n@torch.inference_mode()\ndef evaluate(model, loader, device, class_count):\n    model.eval()\n    actual, predicted, confidence = [], [], []\n    loss = 0.0\n    for images, labels in loader:\n        labels = labels.to(device)\n        logits = model(images.to(device))\n        loss += nn.functional.cross_entropy(logits, labels, reduction=\'sum\').item()\n        conf, pred = logits.softmax(1).max(1)\n        actual.extend(labels.cpu().tolist())\n        predicted.extend(pred.cpu().tolist())\n        confidence.extend(conf.cpu().tolist())\n    return dict(loss=loss / len(actual),\n        macro_f1=float(f1_score(actual, predicted, labels=list(range(class_count)), average=\'macro\', zero_division=0)),\n        accuracy=float(accuracy_score(actual, predicted)), actual=actual, predicted=predicted, confidence=confidence)\n\n\ndef run_one(args, datasets, classes, ssl_state, method, budget, seed, common):\n    output = args.output_dir / method / f\'rotulos_{budget}\' / f\'seed_{seed}\'\n    output.mkdir(parents=True, exist_ok=True)\n    config = dict(**common, method=method, budget=budget, seed=seed)\n    if (output / \'metrics.json\').is_file():\n        metrics = json.loads((output / \'metrics.json\').read_text(encoding=\'utf-8\'))\n        if metrics[\'config\'] != config:\n            raise ValueError(f\'Configuração diferente em {output}; use outra pasta de saída.\')\n        if (output / \'predictions.csv\').is_file() and (output / \'best.pt\').is_file():\n            print(\'Já concluído:\', output, flush=True)\n            return metrics\n\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    # Initialize the classifier identically for each paired seed, independent of backbone loading.\n    classifier = nn.Linear(512, len(classes))\n    model = resnet18(weights=ResNet18_Weights.DEFAULT if method == \'piso\' else None)\n    if method == \'simclr\':\n        model.fc = nn.Identity()\n        model.load_state_dict(ssl_state[\'backbone_state_dict\'], strict=True)\n    model.fc = classifier\n    model.to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)\n    generator = torch.Generator().manual_seed(seed)\n    loaders = {split: DataLoader(ds, batch_size=args.batch_size, shuffle=split == \'train\',\n        num_workers=args.workers, pin_memory=device.type == \'cuda\',\n        generator=generator if split == \'train\' else None) for split, ds in datasets.items()}\n    history, best_f1, best_loss, best_epoch, start = [], -1.0, float(\'inf\'), 0, 1\n    resume_path = output / \'resume.pt\'\n    if resume_path.is_file():\n        saved = torch.load(resume_path, map_location=\'cpu\', weights_only=True)\n        if saved[\'config\'] != config:\n            raise ValueError(f\'Checkpoint incompatível: {resume_path}\')\n        model.load_state_dict(saved[\'model\'])\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        history = saved[\'history\']\n        best_f1, best_loss, best_epoch = saved[\'best_f1\'], saved[\'best_loss\'], saved[\'best_epoch\']\n        start = saved[\'epoch\'] + 1\n        torch.set_rng_state(saved[\'torch_rng\'])\n        generator.set_state(saved[\'loader_rng\'])\n        if device.type == \'cuda\' and saved[\'cuda_rng\']:\n            torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        print(f\'Retomando {method} K={budget} seed={seed} época={start}\', flush=True)\n\n    for epoch in range(start, args.epochs + 1):\n        model.train()\n        train_loss, examples = 0.0, 0\n        for images, labels in loaders[\'train\']:\n            images, labels = images.to(device), labels.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = nn.functional.cross_entropy(model(images), labels)\n            loss.backward()\n            optimizer.step()\n            train_loss += loss.item() * len(labels)\n            examples += len(labels)\n        val = evaluate(model, loaders[\'val\'], device, len(classes))\n        history.append(dict(epoch=epoch, train_loss=train_loss / examples,\n                            val_loss=val[\'loss\'], val_macro_f1=val[\'macro_f1\']))\n        if (val[\'macro_f1\'], -val[\'loss\']) > (best_f1, -best_loss):\n            best_f1, best_loss, best_epoch = val[\'macro_f1\'], val[\'loss\'], epoch\n            torch.save(dict(model_state_dict=model.state_dict(), class_names=classes,\n                image_size=args.image_size, mean=MEAN, std=STD, config=config, best_epoch=epoch), output / \'best.pt\')\n        write_json(output / \'history.json\', history)\n        temporary = output / \'resume.pt.tmp\'\n        torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), config=config,\n            epoch=epoch, history=history, best_f1=best_f1, best_loss=best_loss, best_epoch=best_epoch,\n            torch_rng=torch.get_rng_state(), loader_rng=generator.get_state(),\n            cuda_rng=torch.cuda.get_rng_state_all() if device.type == \'cuda\' else []), temporary)\n        temporary.replace(resume_path)\n        print(f\'{method} K={budget} seed={seed} época {epoch}/{args.epochs}: \'\n              f\'loss treino={train_loss/examples:.4f}, F1 val={val["macro_f1"]:.4f}\', flush=True)\n\n    best = torch.load(output / \'best.pt\', map_location=device, weights_only=True)\n    model.load_state_dict(best[\'model_state_dict\'])\n    test = evaluate(model, loaders[\'test_public\'], device, len(classes))\n    predictions = [dict(image_id=image_id, actual=classes[actual], predicted=classes[pred], confidence=conf)\n        for (image_id, _), actual, pred, conf in zip(datasets[\'test_public\'].rows,\n            test[\'actual\'], test[\'predicted\'], test[\'confidence\'])]\n    write_csv(output / \'predictions.csv\', predictions)\n    per_class = f1_score(test[\'actual\'], test[\'predicted\'], labels=list(range(len(classes))), average=None, zero_division=0)\n    metrics = dict(method=method, budget=budget, seed=seed, macro_f1=test[\'macro_f1\'],\n        accuracy=test[\'accuracy\'], best_epoch=best_epoch, val_macro_f1=best_f1,\n        per_class_f1=per_class.tolist(), config=config)\n    write_json(output / \'metrics.json\', metrics)\n    resume_path.unlink(missing_ok=True)\n    del model, optimizer\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n    return metrics\n\n\ndef summarize(output, results, classes):\n    write_csv(output / \'curva_orcamento.csv\', [dict(metodo=r[\'method\'], orcamento=r[\'budget\'],\n        numero_rotulos=r[\'budget\'] * len(classes), seed=r[\'seed\'], f1_macro=r[\'macro_f1\'],\n        acuracia=r[\'accuracy\'], melhor_epoca=r[\'best_epoch\']) for r in results])\n    summary = []\n    for method in (\'piso\', \'simclr\'):\n        for budget in (1, 5, 20):\n            runs = [r for r in results if r[\'method\'] == method and r[\'budget\'] == budget]\n            if not runs:\n                continue\n            summary.append(dict(metodo=method, orcamento=budget, numero_rotulos=budget * len(classes),\n                seeds=len(runs), f1_media=float(np.mean([r[\'macro_f1\'] for r in runs])),\n                f1_desvio=float(np.std([r[\'macro_f1\'] for r in runs], ddof=1)) if len(runs)>1 else 0.0,\n                acuracia_media=float(np.mean([r[\'accuracy\'] for r in runs])),\n                acuracia_desvio=float(np.std([r[\'accuracy\'] for r in runs], ddof=1)) if len(runs)>1 else 0.0))\n    write_csv(output / \'resumo_orcamento.csv\', summary)\n    fig, ax = plt.subplots(figsize=(9, 5))\n    for method, label in [(\'piso\', \'Piso: ImageNet + fine-tuning\'), (\'simclr\', \'Método: ImageNet + SimCLR + fine-tuning\')]:\n        rows = [r for r in summary if r[\'metodo\'] == method]\n        ax.errorbar([r[\'numero_rotulos\'] for r in rows], [r[\'f1_media\'] for r in rows],\n                    yerr=[r[\'f1_desvio\'] for r in rows], marker=\'o\', capsize=5, label=label)\n    ax.axhline(CEILING, linestyle=\'--\', color=\'gray\', label=\'Teto de referência: 0,686\')\n    ax.set(xscale=\'log\', xlabel=\'Número total de rótulos (escala log)\', ylabel=\'F1 macro no test_public\', ylim=(0, 1))\n    ax.set_xticks([21,105,420], [\'21 (1/classe)\', \'105 (5/classe)\', \'420 (20/classe)\'])\n    ax.grid(alpha=.25)\n    ax.legend(fontsize=8)\n    fig.tight_layout()\n    fig.savefig(output / \'curva_orcamento.png\', dpi=180)\n    plt.close(fig)\n    paired = {method: {r[\'seed\']: r for r in results if r[\'method\'] == method and r[\'budget\'] == 5}\n              for method in (\'piso\', \'simclr\')}\n    seeds = sorted(set(paired[\'piso\']) & set(paired[\'simclr\']))\n    if len(seeds) != 3:\n        return\n    piso = np.array([paired[\'piso\'][s][\'per_class_f1\'] for s in seeds])\n    simclr = np.array([paired[\'simclr\'][s][\'per_class_f1\'] for s in seeds])\n    delta = simclr - piso\n    gains = [dict(classe=name, piso_f1_media=float(piso[:, i].mean()),\n        metodo_f1_media=float(simclr[:, i].mean()), ganho_medio=float(delta[:, i].mean()),\n        ganho_desvio=float(delta[:, i].std(ddof=1)),\n        seeds_com_ganho=int((delta[:, i] > 0).sum())) for i, name in enumerate(classes)]\n    gains.sort(key=lambda r:r[\'ganho_medio\'], reverse=True)\n    write_csv(output / \'ganho_por_classe.csv\', gains)\n    fig, ax = plt.subplots(figsize=(10, 8))\n    ax.barh([r[\'classe\'] for r in gains], [r[\'ganho_medio\'] for r in gains],\n        xerr=[r[\'ganho_desvio\'] for r in gains], color=[\'seagreen\' if r[\'ganho_medio\']>0 else \'indianred\' for r in gains])\n    ax.axvline(0, color=\'black\', linewidth=1)\n    ax.invert_yaxis()\n    ax.set(xlabel=\'Ganho de F1 por classe: SimCLR − piso (média ± desvio, 3 seeds)\', title=\'Orçamento de 5 rótulos por classe\')\n    fig.tight_layout()\n    fig.savefig(output / \'ganho_por_classe.png\', dpi=180)\n    plt.close(fig)\n    lines = [\'# Parte 2 — Curva de orçamento\', \'\',\n        \'Seeds 42, 43 e 44 no orçamento 5; seed 42 nos demais. Desvio amostral (ddof=1).\',\n        \'As seeds alteram o treino, mantendo os CSVs oficiais e o checkpoint SimCLR fixos.\',\n        \'Os dois métodos usam o mesmo pré-processamento, otimizador, taxa de aprendizado e seleção pelo val.\',\n        \'Teto fornecido pelo monitor: 0,686; referência, não limite teórico.\', \'\',\n        \'| Método | Rótulos/classe | F1 macro (média ± desvio) | Acurácia (média ± desvio) |\',\n        \'|---|---:|---:|---:|\']\n    lines += [f"| {r[\'metodo\']} | {r[\'orcamento\']} | {r[\'f1_media\']:.4f} ± {r[\'f1_desvio\']:.4f} | {r[\'acuracia_media\']:.4f} ± {r[\'acuracia_desvio\']:.4f} |" for r in summary]\n    positive = [r for r in gains if r[\'ganho_medio\']>0]\n    nonpositive = [r[\'classe\'] for r in gains if r[\'ganho_medio\']<=0]\n    lines += [\'\', \'Maiores ganhos médios positivos: \' + (\', \'.join(f"{r[\'classe\']} ({r[\'ganho_medio\']:+.4f})" for r in positive[:5]) or \'nenhuma classe\') + \'.\',\n        \'\', \'Sem ganho médio positivo: \' + (\', \'.join(nonpositive) or \'nenhuma\') + \'.\', \'\',\n        \'O ganho é calculado entre seeds pareadas. Três seeds não demonstram significância estatística; classes com poucos exemplos no teste podem variar bastante.\',\n        \'As predições individuais e os F1 de todas as classes ficam nos CSVs/JSONs para auditoria.\']\n    (output / \'RELATORIO.md\').write_text(\'\\n\'.join(lines), encoding=\'utf-8\')\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--data-dir\', type=Path, default=Path(\'pa3_dados\'))\n    parser.add_argument(\'--part1-zip\', type=Path, default=Path(\'parte1trilhaA/resultados/pa3_resultados_completos.zip\'))\n    parser.add_argument(\'--ssl-checkpoint\', type=Path)\n    parser.add_argument(\'--output-dir\', type=Path, default=Path(\'parte2/resultados\'))\n    parser.add_argument(\'--epochs\', type=int, default=30)\n    parser.add_argument(\'--batch-size\', type=int, default=16)\n    parser.add_argument(\'--image-size\', type=int, default=224)\n    parser.add_argument(\'--workers\', type=int, default=0)\n    parser.add_argument(\'--lr\', type=float, default=1e-4)\n    parser.add_argument(\'--check-only\', action=\'store_true\')\n    parser.add_argument(\'--allow-cpu\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.epochs<1 or args.batch_size<2 or args.image_size<32 or args.workers<0 or args.lr<=0:\n        parser.error(\'Argumentos de treino inválidos.\')\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    classes = load_classes(args.data_dir / \'classes.csv\')\n    ssl_path = locate_ssl(args)\n    state = torch.load(ssl_path, map_location=\'cpu\', weights_only=True)\n    assert state[\'initialization\'] == \'imagenet\' and len(state[\'ssl_loss\']) == state[\'epochs\']\n    assert state[\'image_size\'] == args.image_size, \'Mantenha a resolução da Parte 1.\'\n    all_datasets = {}\n    for budget in (1,5,20):\n        ds = dict(train=CameraTrapDataset(args.data_dir/\'images/train\', args.data_dir/f\'labels/rotulos_{budget}.csv\', classes, args.image_size),\n            val=CameraTrapDataset(args.data_dir/\'images/val\', args.data_dir/\'labels/val.csv\', classes, args.image_size),\n            test_public=CameraTrapDataset(args.data_dir/\'images/test_public\', args.data_dir/\'labels/test_public.csv\', classes, args.image_size))\n        assert Counter(label for _,label in ds[\'train\'].rows) == Counter({i:budget for i in range(len(classes))})\n        ids = [{row[0] for row in ds[split].rows} for split in (\'train\',\'val\',\'test_public\')]\n        assert all(not ids[i] & ids[j] for i in range(3) for j in range(i+1,3)), \'Sobreposição entre splits.\'\n        all_datasets[budget] = ds\n    common = dict(epochs=args.epochs, batch_size=args.batch_size, image_size=args.image_size,\n        lr=args.lr, weight_decay=1e-4, ssl_sha256=sha256(ssl_path),\n        label_sha256={p.name:sha256(p) for p in (args.data_dir/\'labels\').glob(\'*.csv\')},\n        classes=classes, selection=\'val_macro_f1_then_val_loss\', seeds=list(SEEDS),\n        ssl_crop=state[\'augmentation\'], device=\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    write_json(args.output_dir/\'config.json\', common)\n    print(\'Pré-checagem OK. GPU:\', torch.cuda.is_available(), \'Checkpoint:\', ssl_path, flush=True)\n    print(\'10 treinos supervisionados; teto =\', CEILING, flush=True)\n    if args.check_only:\n        return\n    if not torch.cuda.is_available() and not args.allow_cpu:\n        raise RuntimeError(\'GPU ausente. Execute no Kaggle, ou use --allow-cpu explicitamente.\')\n    results = []\n    for budget in (1,5,20):\n        for seed in SEEDS if budget == 5 else (42,):\n            for method in (\'piso\',\'simclr\'):\n                results.append(run_one(args, all_datasets[budget], classes, state, method, budget, seed, common))\n                summarize(args.output_dir, results, classes)\n    print(\'Concluído:\', args.output_dir, flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'}

for name, source in SCRIPTS.items():
    target = WORK/'parte0'/name if name == 'part0.py' else WORK/name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
COMMAND = [sys.executable, '-u', 'parte2.py', '--data-dir', str(DATA),
    '--output-dir', str(OUTPUT), '--epochs', '30', '--batch-size', '16',
    '--image-size', '224', '--workers', '2'] + EXTRA_ARGS
subprocess.run(COMMAND + ['--check-only'], check=True)


## Rodar os 10 treinos

Mesma configuração supervisionada nos dois métodos: AdamW, lr=1e-4, weight decay=1e-4,
batch 16 e 30 épocas. Todas as camadas são ajustadas. O pré-processamento é o da Parte 0.
O SimCLR não é repetido: reutilizamos o checkpoint selecionado na Parte 1 pela validação.
O progresso aparece após cada época. Esta execução refaz a seed 42 para manter o protocolo pareado.


In [ ]:
with (OUTPUT/'execucao.log').open('a', encoding='utf-8') as log:
    process = subprocess.Popen(COMMAND, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        exit_code = process.wait()
        if exit_code:
            raise RuntimeError(f'Treino encerrou com código {exit_code}; veja execucao.log.')
    except BaseException:
        process.terminate()
        process.wait()
        raise


In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown, FileLink
display(pd.read_csv(OUTPUT/'resumo_orcamento.csv'))
display(Image(filename=str(OUTPUT/'curva_orcamento.png')))
display(pd.read_csv(OUTPUT/'ganho_por_classe.csv'))
display(Image(filename=str(OUTPUT/'ganho_por_classe.png')))
display(Markdown((OUTPUT/'RELATORIO.md').read_text()))


## Exportar

`curva_orcamento.csv` tem as dez linhas individuais exigidas pelo PA. `resumo_orcamento.csv`
traz média e desvio amostral (ddof=1). `ganho_por_classe.csv` calcula a diferença entre métodos
para cada seed, depois resume as três diferenças. Ganho <= 0 significa sem ganho médio positivo.
O desvio descreve variação entre seeds, não intervalo de confiança.

O ZIP completo contém checkpoints, históricos e predições por imagem. Preserve-o também
para as próximas partes. O pacote de apresentação exclui os pesos.


In [ ]:
import zipfile
light = Path('/kaggle/working/pa3_parte2_relatorio.zip')
with zipfile.ZipFile(light, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT.iterdir():
        if path.is_file() and path.suffix in {'.csv','.png','.md','.json'}:
            archive.write(path, path.name)
shutil.make_archive('/kaggle/working/pa3_parte2_completo', 'zip', OUTPUT)
display(FileLink(str(light)))
display(FileLink('/kaggle/working/pa3_parte2_completo.zip'))
